# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/engyelgamal18/flyrank-ml-internship-engy/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 : Growing vs declining content

The paper found that growing pages were generally longer and younger than declining pages.
My methodology question:
How was the growing or declining label created and were the measurements used to create this label kept separate from the features used in the comparison? I would want to check this to make sure there is no leakage and that the result is only treated as an observed relationship.
### Finding 2: The freshness window
The papaer found that 31-90 day freshness window had the strongest stable growth to decline ratio.
My methodology question:
How many were included in each freshness group and are the groups large enough to support the comparison? I would especially check small groups because the paper shows that the 361+ day group had only one declining page which can make the ratio unstable.


In [1]:
# Simple checks for the two paper findings

# Finding 1: growing vs declining content
growing_words = 3180
declining_words = 2311

growing_age = 184
declining_age = 230

word_difference = (( growing_words - declining_words) / declining_words)*100
age_difference = ((declining_age - growing_age) / declining_age)*100

print("Finding 1 checks:")
print("Growing pages are about", round(word_difference, 1), "% longer.")
print("Growing pages are about", round(age_difference, 1)," %younger.")

# Finding 2: freshness window
freshness_31_90_ratio = 7.88
growing_361_plus = 283
declining_361_plus = 1

ratio_361_plus = growing_361_plus / declining_361_plus


print("\nFinding 2 check:")
print("31_90 day growth_to_decline_ratio:", freshness_31_90_ratio)
print("361+ day growth_to_decline ratio", ratio_361_plus)
print("361+ declining pages:", declining_361_plus)

if declining_361_plus <=1:
  print("The 361+ results is uunstable because the declining sample is very small.")




Finding 1 checks:
Growing pages are about 37.6 % longer.
Growing pages are about 20.0  %younger.

Finding 2 check:
31_90 day growth_to_decline_ratio: 7.88
361+ day growth_to_decline ratio 283.0
361+ declining pages: 1
The 361+ results is uunstable because the declining sample is very small.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [15]:
from huggingface_hub import HfApi

api = HfApi(token=hf_token)

files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

print("Files with 2026-03:")

for file in files:
  if"2026-03" in file:
    print(file)



Files with 2026-03:
fact_content_daily_performance/month=2026-03/data_0.parquet


In [17]:
import pandas as pd
from huggingface_hub import hf_hub_download
from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")

print("HF token avaliable:", hf_token is not None)

march_file = [f for f in files if "2026-03" in f][0]

print("Using file:", march_file)

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename=march_file,
    repo_type="dataset",
    token=hf_token
)

df = pd.read_parquet(march_path)
df["report_data"] = pd.to_datetime(df["report_date"])

print("Rows loaded:", len(df))

HF token avaliable: True
Using file: fact_content_daily_performance/month=2026-03/data_0.parquet


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Rows loaded: 9841378


In [18]:
from sklearn.model_selection import train_test_split

#BEFORE: random split
random_train, random_test = train_test_split(
    df,
    test_size=0.23,
    random_state=42
)

print("random split")
print("Train rows", len(random_train))
print("Test rows:", len(random_test))

# AFTER:time_aware split
time_train = df[df["report_data"] <= "2026-03-24"].copy()
time_test = df[df["report_data"] > "2026-03-24"].copy()

print("\nTime_aware_split:")
print("Train rows:", len(time_train))
print("Test rows:", len(time_test))

random split
Train rows 7577861
Test rows: 2263517

Time_aware_split:
Train rows: 7548489
Test rows: 2292889


In [27]:
def get_score(data):
  x = data.groupby("content_hash_id").agg(
      impressions=("gsc_impressions", "sum"),
      clicks=("gsc_clicks", "sum"),
      position=("gsc_avg_position", "mean")
  ).reset_index()

  x["ctr"] = (x["clicks"] / x["impressions"].replace(0, pd.NA).fillna(0))
  x["position"] = x["position"].fillna(100)

  x["score"] = x["impressions"] * (1-x["ctr"]) * x["position"]

  return x[["content_hash_id", "score"]]



In [32]:
random_a = get_score(random_train)
random_b = get_score(random_test)

time_a = get_score(time_train)
time_b = get_score(time_test)

random_check = random_a.merge(random_b, on="content_hash_id")
time_check = time_a.merge(time_b, on="content_hash_id")

random_corr = random_check["score_x"].corr(
    random_check["score_y"],
    method="spearman"
)

time_corr = time_check["score_x"].corr(
time_check["score_y"],
method="spearman"
)

print("Random split score:", round(random_corr, 3))
print("Time_aware split score:", round(time_corr, 3))

/tmp/ipykernel_594/3363677915.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  x["ctr"] = (x["clicks"] / x["impressions"].replace(0, pd.NA).fillna(0))
/tmp/ipykernel_594/3363677915.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  x["ctr"] = (x["clicks"] / x["impressions"].replace(0, pd.NA).fillna(0))
/tmp/ipykernel_594/3363677915.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.se

Random split score: 0.931
Time_aware split score: 0.895


### Before and adter validation
Before i used a random split and the score was 0.931. After i used a time aware split where earlier March dates were used first and later March dates were used for validation, the score was 0,895. The time aware score is slightly lower but i trust it more because it is closer to how the ranking would be used in practice. This suggest that the random split may give a slightly more optimistic result.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.